In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Qwen2Model, Qwen2Config
from torch.distributions import Categorical
from transformers.cache_utils import DynamicCache

class VisualEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.in_channels = config["in_channels"]
        self.latent_dim = config["latent_dim"]
        self.input_hw = (config["grid_shape_x"], config["grid_shape_y"])
        
        layers = []
        prev_c = self.in_channels
        for i, out_c in enumerate(config["channels"]):
            layers.append(nn.Conv2d(
                prev_c, out_c, 
                kernel_size=config["kernel_size"], 
                padding=config["padding"], 
                stride=config["stride"] if i == len(config["channels"]) - 1 else 1
            ))
            layers.append(nn.ReLU())
            prev_c = out_c
            
        self.cnn = nn.Sequential(*layers)
        self._cnn_out_dim = self._infer_cnn_out_dim()
        self.fc = nn.Linear(self._cnn_out_dim, self.latent_dim)

    def _infer_cnn_out_dim(self):
        with torch.no_grad():
            dummy = torch.zeros(1, self.in_channels, *self.input_hw)
            out = self.cnn(dummy)
            return out.numel()

    def forward(self, x):
        # Handle both (B, C, H, W) and (B, T, C, H, W)  
        if x.dim() == 5:
            B, T, C, H, W = x.shape
            x = x.view(B * T, C, H, W)
            merge_time = True
        else:
            merge_time = False

        x = self.cnn(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)

        if merge_time:
            x = x.view(B, T, -1)
        return x


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Matias/Documents/GitHub/curry/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Matias/Documents/GitHub/curry/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/Matias/Documents/GitHub/curry/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", lin

In [2]:
class ActionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.model_name = config["model_name"]
        self.hidden_size = config["hidden_size"]
        self.vocab_size = config["vocab_size"]
        self.num_think_steps = config["num_think_steps"]
        
        if self.model_name == "qwen2":
            from transformers import Qwen2Model, Qwen2Config
            qwen_config = Qwen2Config(
                hidden_size=self.hidden_size,
                num_hidden_layers=config["args"]["num_layers"],
                num_attention_heads=config["args"]["num_attention_heads"],
                num_key_value_heads=config["args"]["num_attention_heads"],
                intermediate_size=self.hidden_size * 4,
                vocab_size=self.vocab_size
            )
            self.backbone = Qwen2Model(qwen_config)
        
        # PPO Heads
        self.actor_head = nn.Linear(self.hidden_size, self.vocab_size)
        self.value_head = nn.Linear(self.hidden_size, 1)

    def forward_step(self, x_t, past_key_values=None):
        h = x_t
        current_kv = past_key_values

        # Thinking Loop (Iterative Refinement)
        outputs = self.backbone(
            inputs_embeds=h,
            past_key_values=current_kv,
            use_cache=True
        )
        
        h_out = outputs.last_hidden_state # (B, 1, D)
        new_kv = outputs.past_key_values  # Tuple of KVs
        
        # Heads
        logits = self.actor_head(h_out)
        value = self.value_head(h_out)
        
        return {
            "logits": logits,      # (B, 1, Vocab)
            "value": value,        # (B, 1, 1)
            "hidden_state": h_out, # (B, 1, D)
            "past_key_values": new_kv
        }

    def forward_parallel(self, latents):
        """
        Full sequence parallel pass for Target Generation.
        latents: (B, T, D)
        """
        # Standard Causal Forward Pass
        outputs = self.backbone(
            inputs_embeds=latents,
            output_hidden_states=True,
            return_dict=True
        )
        
        h_seq = outputs.last_hidden_state # (B, T, D)
        
        logits = self.actor_head(h_seq)
        values = self.value_head(h_seq)

        return {
            "logits": logits,
            "values": values,
            "hidden_states": h_seq
        }

In [10]:
config_action_decoder = {
    "model_name": "qwen2",
    "hidden_size": 32,
    "vocab_size": 10,
    "num_think_steps": 1,
    "args": {
        "num_layers": 2,         
        "num_attention_heads": 4, 
    }
}

decoder = ActionDecoder(config_action_decoder)

x_t = torch.randn(4, 1, 32)
past_kv = None # First step has no history


# --- Test in your notebook ---
print("--- Testing Forward Step ---")
with torch.no_grad():
    out = decoder.forward_step(x_t, past_key_values=None)
    current_cache = out["past_key_values"]
for i in current_cache.to_legacy_cache():
    for j in i:
        print(j.shape)# (B, num_heads, seq_len, head_dim) in this case ([4, 4, 1, 8])



--- Testing Forward Step ---
torch.Size([4, 4, 1, 8])
torch.Size([4, 4, 1, 8])
torch.Size([4, 4, 1, 8])
torch.Size([4, 4, 1, 8])


In [11]:
# generate random kv configuration for two layers

k_first = torch.randn(4, 4, 3, 8)
v_first = torch.randn(4, 4, 3, 8)  

k_second = torch.randn(4, 4, 3, 8)
v_second = torch.randn(4, 4, 3, 8)

past_key_values = (
    (k_first, v_first),
    (k_second, v_second)
)

# test forward step with past kvs
print("\n--- Testing Forward Step with Past KVs ---")
with torch.no_grad():
    out = decoder.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values))
    current_cache = out["past_key_values"]
for i in current_cache.to_legacy_cache():
    for j in i:
        print(j.shape)  # (B, num_heads, seq_len, head_dim) in this case ([4, 4, 4, 8])


--- Testing Forward Step with Past KVs ---
torch.Size([4, 4, 4, 8])
torch.Size([4, 4, 4, 8])
torch.Size([4, 4, 4, 8])
torch.Size([4, 4, 4, 8])


In [ ]:
# test training forward step for two different kv configurations

#make optimizer
optimizer = torch.optim.Adam(decoder.parameters(), lr=1e-3)
decoder.train()

k_first_2 = torch.randn(4, 4, 3, 8)
v_first_2 = torch.randn(4, 4, 3, 8)
k_second_2 = torch.randn(4, 4, 3, 8)
v_second_2 = torch.randn(4, 4, 3, 8)
past_key_values_2 = (
    (k_first_2, v_first_2),
    (k_second_2, v_second_2)
)
target_latent1 = torch.randn(4, 1, 32)
target_latent2 = torch.randn(4, 1, 32)

print("\n--- Testing backward Step with Different Past KVs ---")
x_t.requires_grad_(True)
for i in range(101):
    optimizer.zero_grad()
    out1 = decoder.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values))
    out2 = decoder.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values_2))

    #Calculate MSE loss
    loss1 = F.mse_loss(out1["hidden_state"], target_latent1)
    loss2 = F.mse_loss(out2["hidden_state"], target_latent2)
    if i%10 ==  0:
        print(f"Iteration {i}, Loss1: {loss1.item()}, Loss2: {loss2.item()}")

    total_loss = loss1 + loss2

    total_loss.backward()
    optimizer.step()
"""
Iteration 0, Loss1: 2.0995302200317383, Loss2: 2.0077216625213623
...
Iteration 100, Loss1: 0.0010840415488928556, Loss2: 0.017802512273192406
"""


print("\n---Moving from model learning to kv learning---")


#On the other hand we'd like to know if we can freeze the decoder and train the cache.
k_first = torch.randn(4, 4, 3, 8)
v_first = torch.randn(4, 4, 3, 8)  

k_second = torch.randn(4, 4, 3, 8)
v_second = torch.randn(4, 4, 3, 8)

past_key_values = (
    (nn.Parameter(k_first), nn.Parameter(v_first)),
    (nn.Parameter(k_second), nn.Parameter(v_second))
)

k_first_2 = torch.randn(4, 4, 3, 8)
v_first_2 = torch.randn(4, 4, 3, 8)
k_second_2 = torch.randn(4, 4, 3, 8)
v_second_2 = torch.randn(4, 4, 3, 8)
past_key_values_2 = (
    (nn.Parameter(k_first_2), nn.Parameter(v_first_2)),
    (nn.Parameter(k_second_2), nn.Parameter(v_second_2))
)

params = (
    p for kv_tuple in (past_key_values, past_key_values_2)
    for kv_pair in kv_tuple
    for p in kv_pair
)

optimizer_kv = torch.optim.Adam(params, lr=1e-2)

for i in range(101):
    optimizer_kv.zero_grad()
    out1 = decoder.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values))
    out2 = decoder.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values_2))

    #Calculate MSE loss
    loss1 = F.mse_loss(out1["hidden_state"], target_latent1)
    loss2 = F.mse_loss(out2["hidden_state"], target_latent2)
    if i%10 == 0:
        print(f"Iteration {i}, Loss1: {loss1.item()}, Loss2: {loss2.item()}")

    total_loss = loss1 + loss2

    total_loss.backward()
    optimizer_kv.step()

"""
Iteration 0, Loss1: 0.5585730075836182, Loss2: 0.9463440179824829
...
Iteration 100, Loss1: 0.014445982873439789, Loss2: 0.03492976352572441

:nn.Parameter is learnt slower but can be learnt if attention allows it
"""
print("done!")


--- Testing backward Step with Different Past KVs ---
Iteration 0, Loss1: 2.019200325012207, Loss2: 1.9979088306427002
Iteration 10, Loss1: 1.1264880895614624, Loss2: 0.9897773265838623
Iteration 20, Loss1: 0.5261551141738892, Loss2: 0.49007391929626465
Iteration 30, Loss1: 0.24971044063568115, Loss2: 0.19460532069206238
Iteration 40, Loss1: 0.0747399851679802, Loss2: 0.06753544509410858
Iteration 50, Loss1: 0.02389467880129814, Loss2: 0.03139618784189224
Iteration 60, Loss1: 0.01541624404489994, Loss2: 0.023681672289967537
Iteration 70, Loss1: 0.01342688873410225, Loss2: 0.021721726283431053
Iteration 80, Loss1: 0.01195596158504486, Loss2: 0.02022758312523365
Iteration 90, Loss1: 0.011296906508505344, Loss2: 0.019282124936580658
Iteration 100, Loss1: 0.010987011715769768, Loss2: 0.018652673810720444

---Moving from model learning to kv learning---
Iteration 0, Loss1: 1.1737724542617798, Loss2: 0.6963146924972534
Iteration 10, Loss1: 0.8075807094573975, Loss2: 0.46343305706977844
Iter

In [29]:
#Time to check if gradient flows through the kv cache at generation time
config_action_decoder = {
    "model_name": "qwen2",
    "hidden_size": 32,
    "vocab_size": 10,
    "num_think_steps": 1,
    "args": {
        "num_layers": 2,         
        "num_attention_heads": 4, 
    }
}

decoder_1 = ActionDecoder(config_action_decoder)
decoder_2 = ActionDecoder(config_action_decoder)

optimizer_1 = torch.optim.Adam(decoder_1.parameters(), lr=1e-3)
optimizer_2 = torch.optim.Adam(decoder_2.parameters(), lr=1e-3)

x_t = torch.randn(4, 1, 32)
y_t = torch.randn(4, 1, 32)
k_first = torch.randn(4, 4, 3, 8)
v_first = torch.randn(4, 4, 3, 8)  

k_second = torch.randn(4, 4, 3, 8)
v_second = torch.randn(4, 4, 3, 8)
target_latent = torch.randn(4, 1, 32)
past_key_values = (
    (nn.Parameter(k_first), nn.Parameter(v_first)),
    (nn.Parameter(k_second), nn.Parameter(v_second))
)
print("\n---Beggining test of gradient leakage---")
for i in range(101):
    optimizer_1.zero_grad()

    out1 = decoder_1.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values))
    out1 = decoder_1.forward_step(y_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values))
    new_past_key_values = (
        (out1["past_key_values"][0][0][:,:,-2:,:], out1["past_key_values"][0][1][:,:,-2:,:]),
        (out1["past_key_values"][0][1][:,:,-2:,:], out1["past_key_values"][1][1][:,:,-2:,:])
    )
    out2 = decoder.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(new_past_key_values))
    loss2 = F.mse_loss(out2["hidden_state"], target_latent)
    if i%10 == 0:
        print(f"Iteration {i}, Loss: {loss2.item()}")
    """
    Iteration 0, Loss: 1.798529028892517

    ...
    Iteration 99, Loss: 0.9148995876312256
    Gradient travels, this has to be stopped!
    """
    loss2.backward()
    optimizer_1.step()


decoder_1 = ActionDecoder(config_action_decoder)
decoder_2 = ActionDecoder(config_action_decoder)

optimizer_1 = torch.optim.Adam(decoder_1.parameters(), lr=1e-3)
optimizer_2 = torch.optim.Adam(decoder_2.parameters(), lr=1e-3)

x_t = torch.randn(4, 1, 32)
y_t = torch.randn(4, 1, 32)
k_first = torch.randn(4, 4, 3, 8)
v_first = torch.randn(4, 4, 3, 8)  

k_second = torch.randn(4, 4, 3, 8)
v_second = torch.randn(4, 4, 3, 8)
target_latent = torch.randn(4, 1, 32)
past_key_values = (
    (nn.Parameter(k_first), nn.Parameter(v_first)),
    (nn.Parameter(k_second), nn.Parameter(v_second))
)
print("\n---Begining second test of gradient leakage---")
for i in range(101):
    optimizer_1.zero_grad()

    out1 = decoder_1.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values))
    out1 = decoder_1.forward_step(y_t, past_key_values=DynamicCache.from_legacy_cache(past_key_values))
    new_past_key_values = (
        (out1["past_key_values"][0][0][:,:,-2:,:].detach(), out1["past_key_values"][0][1][:,:,-2:,:].detach()),
        (out1["past_key_values"][0][1][:,:,-2:,:].detach(), out1["past_key_values"][1][1][:,:,-2:,:].detach())
    )
    out2 = decoder.forward_step(x_t, past_key_values=DynamicCache.from_legacy_cache(new_past_key_values))
    loss2 = F.mse_loss(out2["hidden_state"], target_latent)
    if i%10 == 0:
        print(f"Iteration {i}, Loss: {loss2.item()}")
    """
    Iteration 0, Loss: 2.2760303020477295
    ...
    Iteration 100, Loss: 2.2760303020477295
    Succes!!!
    """
    loss2.backward()
    optimizer_1.step()


---Beggining test of gradient leakage---
Iteration 0, Loss: 1.4571096897125244
Iteration 10, Loss: 1.405897617340088
Iteration 20, Loss: 1.3479750156402588
Iteration 30, Loss: 1.2788888216018677
Iteration 40, Loss: 1.2016385793685913
Iteration 50, Loss: 1.1187478303909302
Iteration 60, Loss: 1.0333093404769897
Iteration 70, Loss: 0.9494779109954834
Iteration 80, Loss: 0.8710395693778992
Iteration 90, Loss: 0.800230085849762
Iteration 100, Loss: 0.7375583648681641

---Begining second test of gradient leakage---
Iteration 0, Loss: 2.2760303020477295
Iteration 10, Loss: 2.2760303020477295
Iteration 20, Loss: 2.2760303020477295
Iteration 30, Loss: 2.2760303020477295
Iteration 40, Loss: 2.2760303020477295
Iteration 50, Loss: 2.2760303020477295
Iteration 60, Loss: 2.2760303020477295
Iteration 70, Loss: 2.2760303020477295
Iteration 80, Loss: 2.2760303020477295
Iteration 90, Loss: 2.2760303020477295
Iteration 100, Loss: 2.2760303020477295
